In [16]:
import glob, os

for f in glob.glob("output/cache/gtrends_term_*.csv"):
    os.remove(f)

print("Cleared cached gtrends term files.")

Cleared cached gtrends term files.


In [17]:
# =========================
# Notebook 3 (Manual): Build a Google Trends download plan
# Assumes Notebooks 1-2 ran and created: output/final/event_panel_yf.csv
#
# Outputs:
#   output/final/gtrends_download_plan.csv
#   output/final/gtrends_download_instructions.txt
#   output/figures/ticker_liquidity_rank.png
# =========================

import os
import time
import random
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

BASE_DIR = "output"
FINAL_DIR = os.path.join(BASE_DIR, "final")
FIG_DIR = os.path.join(BASE_DIR, "figures")

for d in [BASE_DIR, FINAL_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

def progress(msg):
    print(msg, flush=True)

def sleep_polite(base=0.8, jitter=0.8):
    time.sleep(base + random.random() * jitter)

# -------------------------
# Step 1: Load event panel tickers from Notebook 2
# -------------------------
events_path = os.path.join(FINAL_DIR, "event_panel_yf.csv")
if not os.path.exists(events_path):
    raise FileNotFoundError(
        f"Missing {events_path}. Run Notebook 2 first."
    )

events = pd.read_csv(events_path, parse_dates=["earnings_datetime", "event_day"])
events["ticker"] = events["ticker"].astype(str).str.upper().str.strip()
tickers = sorted(events["ticker"].dropna().unique().tolist())
progress(f"Loaded {len(tickers)} tickers from event panel.")

# -------------------------
# Step 2: Company name mapping
# You can add more if your event panel includes other tickers.
# -------------------------
TICKER_TO_COMPANY = {
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "AVGO": "Broadcom",
    "BAC": "Bank of America",
    "COST": "Costco",
    "GOOGL": "Google",
    "HD": "Home Depot",
    "JPM": "JPMorgan",
    "MA": "Mastercard",
    "META": "Meta",
    "MSFT": "Microsoft",
    "NVDA": "Nvidia",
    "PG": "Procter Gamble",
    "TSLA": "Tesla",
    "UNH": "UnitedHealth",
    "XOM": "Exxon",
}

# Fill missing company names from yfinance if possible
# This makes your Google Trends terms higher quality
company_rows = []
for i, t in enumerate(tickers):
    progress(f"[company {i+1}/{len(tickers)}] {t}")
    name = TICKER_TO_COMPANY.get(t)
    try:
        if name is None or str(name).strip() == "":
            tk = yf.Ticker(t)
            info = tk.get_info()
            long_name = info.get("longName") or info.get("shortName")
            if long_name:
                name = long_name.split(",")[0].strip()
    except Exception:
        pass

    company_rows.append({"ticker": t, "company_name": name})
    sleep_polite()

company_map = pd.DataFrame(company_rows)

# -------------------------
# Step 3: Rank tickers by "likely searchable + meaningful" heuristics
# We want tickers that are:
#   large, liquid, and consumer-visible
# Because those are most likely to have Trends volume and clean attention variation.
# -------------------------
meta_rows = []
for i, t in enumerate(tickers):
    progress(f"[meta {i+1}/{len(tickers)}] {t}")
    try:
        tk = yf.Ticker(t)
        fi = tk.fast_info
        last = fi.get("last_price", np.nan)
        mcap = fi.get("market_cap", np.nan)
        vol10 = fi.get("ten_day_average_volume", np.nan)
        dollar_vol = np.nan
        if pd.notna(last) and pd.notna(vol10):
            dollar_vol = float(last) * float(vol10)
        meta_rows.append({
            "ticker": t,
            "market_cap": mcap,
            "last_price": last,
            "ten_day_avg_volume": vol10,
            "avg_dollar_volume_10d": dollar_vol
        })
    except Exception:
        meta_rows.append({
            "ticker": t,
            "market_cap": np.nan,
            "last_price": np.nan,
            "ten_day_avg_volume": np.nan,
            "avg_dollar_volume_10d": np.nan
        })
    sleep_polite()

meta = pd.DataFrame(meta_rows)

# Merge
df = meta.merge(company_map, on="ticker", how="left")

# Rank: primarily dollar volume, secondarily market cap
df["rank_dollar_vol"] = df["avg_dollar_volume_10d"].rank(ascending=False, method="min")
df["rank_mcap"] = df["market_cap"].rank(ascending=False, method="min")

df["composite_rank"] = 0.7 * df["rank_dollar_vol"].fillna(df["rank_dollar_vol"].max() + 1) + \
                       0.3 * df["rank_mcap"].fillna(df["rank_mcap"].max() + 1)

df = df.sort_values("composite_rank").reset_index(drop=True)

# Choose shortlist size
SHORTLIST_N = min(12, len(df))
shortlist = df.head(SHORTLIST_N).copy()
progress(f"Shortlisting {SHORTLIST_N} tickers for manual Google Trends downloads.")

# -------------------------
# Step 4: Build the exact Google Trends queries to download
# We provide multiple options per ticker.
# You only need to download ONE series per ticker, but we give fallback terms.
# -------------------------
def build_terms(ticker, company_name):
    company_name = (company_name or "").strip()
    terms = []

    if company_name:
        # Most reliable terms first
        terms.append(f"{company_name} stock")
        terms.append(f"{company_name} earnings")
        terms.append(f"{company_name} earnings date")

    # Ticker-based fallbacks
    terms.append(f"{ticker} stock")
    terms.append(f"{ticker} earnings")
    terms.append(f"{ticker} earnings date")

    # De-duplicate
    seen = set()
    out = []
    for x in terms:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

plans = []
for _, r in shortlist.iterrows():
    t = r["ticker"]
    c = r["company_name"]
    terms = build_terms(t, c)
    plans.append({
        "ticker": t,
        "company_name": c,
        "primary_term_to_try": terms[0],
        "fallback_term_2": terms[1] if len(terms) > 1 else "",
        "fallback_term_3": terms[2] if len(terms) > 2 else "",
        "fallback_term_4": terms[3] if len(terms) > 3 else "",
        "all_terms_list": " | ".join(terms)
    })

plan = pd.DataFrame(plans)

# -------------------------
# Step 5: Write explicit download instructions
# You can choose your time range and location.
# I recommend "Past 5 years" and "United States" first, then Worldwide if too sparse.
# -------------------------
instructions = []
instructions.append("Google Trends Manual Download Plan")
instructions.append("")
instructions.append("Recommended settings to start with for consistency across firms:")
instructions.append("Location: United States")
instructions.append("Time range: Past 5 years")
instructions.append("Search type: Web Search")
instructions.append("Category: All categories")
instructions.append("")
instructions.append("If a term shows 'not enough data' in US, try:")
instructions.append("1) Switch Location to Worldwide")
instructions.append("2) Try the fallback terms for the same ticker")
instructions.append("3) If still sparse, reduce the window to Past 12 months")
instructions.append("")
instructions.append("What to download for each ticker:")
instructions.append("Download the CSV from the download icon on the Interest over time chart.")
instructions.append("Name the file like this so merging is easy later:")
instructions.append("gtrends_<TICKER>_<GEO>_<WINDOW>.csv")
instructions.append("Examples:")
instructions.append("gtrends_AAPL_US_5y.csv")
instructions.append("gtrends_AMZN_WW_5y.csv")
instructions.append("")
instructions.append("Columns expected in the CSV:")
instructions.append("Week (or Day) and a single search interest column for your chosen term.")
instructions.append("")
instructions.append("Ticker list and terms are saved in output/final/gtrends_download_plan.csv")

txt_path = os.path.join(FINAL_DIR, "gtrends_download_instructions.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(instructions))

# -------------------------
# Step 6: Save plan
# -------------------------
plan_path = os.path.join(FINAL_DIR, "gtrends_download_plan.csv")
plan.to_csv(plan_path, index=False)

progress(f"Saved download plan to {plan_path}")
progress(f"Saved instructions to {txt_path}")

# -------------------------
# Step 7: Simple plot so you can justify ticker choice in the report
# -------------------------
plot_df = df.head(min(20, len(df))).copy()
plot_df["log_dollar_vol"] = np.log(plot_df["avg_dollar_volume_10d"].replace(0, np.nan))

plt.figure(figsize=(8, 4))
plt.bar(plot_df["ticker"], plot_df["log_dollar_vol"])
plt.title("Top tickers by liquidity (log 10-day avg dollar volume)")
plt.xlabel("Ticker")
plt.ylabel("log(avg dollar volume)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "ticker_liquidity_rank.png"), dpi=200)
plt.close()

progress("Saved plot output/figures/ticker_liquidity_rank.png")

# Display the plan in notebook
plan

Loaded 16 tickers from event panel.
[company 1/16] AAPL
[company 2/16] AMZN
[company 3/16] AVGO
[company 4/16] BAC
[company 5/16] COST
[company 6/16] GOOGL
[company 7/16] HD
[company 8/16] JPM
[company 9/16] MA
[company 10/16] META
[company 11/16] MSFT
[company 12/16] NVDA
[company 13/16] PG
[company 14/16] TSLA
[company 15/16] UNH
[company 16/16] XOM
[meta 1/16] AAPL
[meta 2/16] AMZN
[meta 3/16] AVGO
[meta 4/16] BAC
[meta 5/16] COST
[meta 6/16] GOOGL
[meta 7/16] HD
[meta 8/16] JPM
[meta 9/16] MA
[meta 10/16] META
[meta 11/16] MSFT
[meta 12/16] NVDA
[meta 13/16] PG
[meta 14/16] TSLA
[meta 15/16] UNH
[meta 16/16] XOM
Shortlisting 12 tickers for manual Google Trends downloads.
Saved download plan to output/final/gtrends_download_plan.csv
Saved instructions to output/final/gtrends_download_instructions.txt
Saved plot output/figures/ticker_liquidity_rank.png


,ticker,company_name,primary_term_to_try,fallback_term_2,fallback_term_3,fallback_term_4,all_terms_list
0,AAPL,Apple,Apple stock,Apple earnings,Apple earnings date,AAPL stock,Apple stock | Apple earnings | Apple earnings ...
1,AMZN,Amazon,Amazon stock,Amazon earnings,Amazon earnings date,AMZN stock,Amazon stock | Amazon earnings | Amazon earnin...
2,AVGO,Broadcom,Broadcom stock,Broadcom earnings,Broadcom earnings date,AVGO stock,Broadcom stock | Broadcom earnings | Broadcom ...
3,BAC,Bank of America,Bank of America stock,Bank of America earnings,Bank of America earnings date,BAC stock,Bank of America stock | Bank of America earnin...
4,COST,Costco,Costco stock,Costco earnings,Costco earnings date,COST stock,Costco stock | Costco earnings | Costco earnin...
5,GOOGL,Google,Google stock,Google earnings,Google earnings date,GOOGL stock,Google stock | Google earnings | Google earnin...
6,HD,Home Depot,Home Depot stock,Home Depot earnings,Home Depot earnings date,HD stock,Home Depot stock | Home Depot earnings | Home ...
7,JPM,JPMorgan,JPMorgan stock,JPMorgan earnings,JPMorgan earnings date,JPM stock,JPMorgan stock | JPMorgan earnings | JPMorgan ...
8,MA,Mastercard,Mastercard stock,Mastercard earnings,Mastercard earnings date,MA stock,Mastercard stock | Mastercard earnings | Maste...
9,META,Meta,Meta stock,Meta earnings,Meta earnings date,META stock,Meta stock | Meta earnings | Meta earnings dat...


In [19]:
import os
import re
import glob
import pandas as pd
import numpy as np

MANUAL_DIR = "output/manual"
FINAL_DIR = "output/final"
TABLE_DIR = "output/tables"

os.makedirs(FINAL_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

# Tickers you actually downloaded (safe to leave extras)
KNOWN_TICKERS = [
    "AAPL","AMZN","GOOGL","MSFT","META","NVDA","TSLA",
    "AVGO","BAC","COST","HD","JPM","MA","PG","UNH","XOM"
]

COMPANY_FALLBACK = {
    "APPLE": "AAPL",
    "AMAZON": "AMZN",
    "GOOGLE": "GOOGL",
    "MICROSOFT": "MSFT",
    "META": "META",
    "NVIDIA": "NVDA",
    "TESLA": "TSLA",
    "BROADCOM": "AVGO",
    "BANK OF AMERICA": "BAC",
    "COSTCO": "COST",
    "HOME DEPOT": "HD",
    "JPMORGAN": "JPM",
    "MASTERCARD": "MA",
    "PROCTER": "PG",
    "UNITEDHEALTH": "UNH",
    "EXXON": "XOM",
}

def infer_ticker_from_filename(path: str):
    base = os.path.basename(path).upper()
    for t in KNOWN_TICKERS:
        if re.search(rf"\b{t}\b", base):
            return t
    for k, v in COMPANY_FALLBACK.items():
        if k in base:
            return v
    return None

def infer_term_from_filename(path: str):
    return os.path.splitext(os.path.basename(path))[0].strip()

def read_trends_weekly_csv(path: str) -> pd.DataFrame:
    """
    Matches your files:
      Category: All categories
      <blank>
      Week,<term>: (Worldwide)
      YYYY-MM-DD,value
    """
    # Use utf-8-sig to tolerate BOMs if any
    df = pd.read_csv(path, encoding="utf-8-sig", skiprows=2)
    df.columns = [str(c).strip().replace("\ufeff", "") for c in df.columns]

    if df.shape[1] < 2:
        return pd.DataFrame(columns=["date", "SVI"])

    date_col = df.columns[0]
    val_col = df.columns[1]

    out = df[[date_col, val_col]].rename(columns={date_col: "date", val_col: "SVI"}).copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.dropna(subset=["date"])

    # Your files are numeric already; keep this robust anyway
    out["SVI"] = (
        out["SVI"]
        .astype(str)
        .str.replace("<", "", regex=False)
        .str.strip()
    )
    out["SVI"] = pd.to_numeric(out["SVI"], errors="coerce").fillna(0.0)

    return out

def to_weekly_wmon(df: pd.DataFrame) -> pd.DataFrame:
    # Normalize everything to weekly Monday frequency
    s = df.sort_values("date").set_index("date")["SVI"]
    wk = s.resample("W-MON").mean()
    return wk.reset_index().rename(columns={"SVI": "SVI_weekly"})

def zscore(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    sd = x.std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.zeros(len(x)), index=x.index)
    return (x - x.mean()) / sd

# -----------------------------
# Read all manual CSVs
# -----------------------------
paths = sorted(glob.glob(os.path.join(MANUAL_DIR, "*.csv")))
if not paths:
    raise FileNotFoundError(f"No CSVs found in {MANUAL_DIR}. Put your Trends CSVs there first.")

term_rows = []
audit_rows = []

for p in paths:
    fname = os.path.basename(p)
    ticker = infer_ticker_from_filename(p)
    term_label = infer_term_from_filename(p)

    try:
        if ticker is None:
            audit_rows.append({"file": fname, "status": "ticker_not_inferred", "ticker": None, "term": term_label})
            continue

        raw = read_trends_weekly_csv(p)
        if raw.empty:
            audit_rows.append({"file": fname, "status": "empty_after_read", "ticker": ticker, "term": term_label})
            continue

        wk = to_weekly_wmon(raw)
        wk["ticker"] = ticker
        wk["term"] = term_label
        term_rows.append(wk)

        audit_rows.append({"file": fname, "status": "ok", "ticker": ticker, "term": term_label, "n_weeks": int(wk.shape[0])})

    except Exception as e:
        audit_rows.append({"file": fname, "status": f"error: {type(e).__name__}", "ticker": ticker, "term": term_label})

audit = pd.DataFrame(audit_rows)
audit_path = os.path.join(TABLE_DIR, "gtrends_merge_audit.csv")
audit.to_csv(audit_path, index=False)

if not term_rows:
    raise RuntimeError(f"No usable series were read. Open {audit_path} and check the status column.")

terms = pd.concat(term_rows, ignore_index=True)
terms = terms.sort_values(["ticker", "term", "date"]).reset_index(drop=True)

# Same units across all terms: z-score within (ticker, term)
terms["SVI_z"] = terms.groupby(["ticker", "term"])["SVI_weekly"].transform(zscore)

# Aggregate multiple terms into one ticker-level attention index
panel = (
    terms.groupby(["ticker", "date"], as_index=False)
    .agg(
        attention_index_z=("SVI_z", "mean"),
        n_terms_used=("term", "nunique"),
    )
).sort_values(["ticker", "date"]).reset_index(drop=True)

# Optional baseline and abnormal attention (useful for your report)
panel["attention_ma_8"] = panel.groupby("ticker")["attention_index_z"].transform(lambda x: x.rolling(8).mean())
panel["attention_abn"] = panel["attention_index_z"] - panel["attention_ma_8"]

terms_out = os.path.join(FINAL_DIR, "attention_panel_gtrends_terms.csv")
panel_out = os.path.join(FINAL_DIR, "attention_panel_gtrends.csv")

terms.to_csv(terms_out, index=False)
panel.to_csv(panel_out, index=False)

print(f"Saved audit to: {audit_path}")
print(f"Saved term-level panel to: {terms_out}")
print(f"Saved ticker-level attention panel to: {panel_out}")
print("Tickers included:", sorted(panel["ticker"].unique().tolist()))
print("Date range:", panel["date"].min(), "to", panel["date"].max())

Saved audit to: output/tables/gtrends_merge_audit.csv
Saved term-level panel to: output/final/attention_panel_gtrends_terms.csv
Saved ticker-level attention panel to: output/final/attention_panel_gtrends.csv
Tickers included: ['AAPL', 'AMZN', 'AVGO', 'BAC', 'COST', 'GOOGL', 'HD', 'JPM', 'MA', 'META', 'MSFT', 'NVDA']
Date range: 2020-12-21 00:00:00 to 2025-12-22 00:00:00
